# Generate Final Dataset

## Import Libraries

In [15]:
import numpy as np
import pandas as pd
import random

## Global Configuration

In [16]:
pd.set_option('display.max_columns', None)

## Load Datasets

In [17]:
dataset_workout = pd.read_csv('../../data/dataset_workout.csv')

In [18]:
df = pd.read_csv('../../data/dataset_users.csv')
df_weekly_progress = pd.read_csv('../../data/dataset_users.csv')

df_merged = pd.merge(
    df,                       # Left Table (Profile Data)
    df_weekly_progress,       # Right Table (Weekly Data)
    on='User_ID',             # Key for connecting
    how='left'                # Join Type
)

In [19]:
SPORTS_COLUMNS = [
    'Badminton', 'Football', 'Basketball', 
    'Volleyball', 'Swim'
]

In [20]:
MUSCLE_GROUPS = {
    'Chest': ['pectorals', 'chest', 'serratus', 'push up', 'press'],
    'Shoulders': ['delts', 'shoulders', 'rotator', 'press', 'raise'],
    'Triceps': ['triceps', 'extension', 'dip', 'pushdown', 'skullcrusher'],
    'Back': ['lats', 'latissimus', 'trapezius', 'row', 'pull', 'chin', 'superman'],
    'Biceps': ['biceps', 'curl', 'hammer'],
    'Quads': ['quads', 'quadriceps', 'squat', 'lunge', 'step up', 'leg press', 'goblet'], 
    'Hamstrings': ['hamstrings', 'curl', 'glute ham', 'deadlift', 'hinge', 'good morning'], 
    'Glutes': ['glutes', 'hip', 'butt', 'bridge', 'kickback', 'thrust'], 
    'Calves': ['calves', 'raise', 'jump'],
    'Abs': ['abs', 'abdominal', 'core', 'crunch', 'plank', 'sit up', 'leg raise', 'twist'],
    'Cardio': ['cardio', 'run', 'jump', 'burpee', 'climber', 'jack', 'sprint'],
    
    # Fallback Categories
    'Push_General': ['chest', 'shoulders', 'triceps', 'push'],
    'Pull_General': ['back', 'biceps', 'pull'],
    'Legs_General': ['legs', 'lower body', 'squat', 'lunge'],
}

## Helper Functions

In [21]:
def clean_string_data(text):
    """
    Parses and cleans string representations of lists from CSV data.

    This utility function is designed to fix formatting artifacts (brackets `[]` and 
    single quotes `'`) that occur when Python lists are saved directly into CSV files 
    as text strings.

    ---------------------------------------------------------------------------
    Args:
        text (str, float, or None): 
            The raw input string to be cleaned (e.g., "['Cable Machine']").
            Also handles NaN (Not a Number) values from Pandas.

    Returns:
        str: A clean, trimmed string (e.g., "Cable Machine").
             Returns "None" if the input was missing or NaN.
    ---------------------------------------------------------------------------
    Example:
        >>> clean_string_data("['Dumbbell']")
        "Dumbbell"
        >>> clean_string_data(np.nan)
        "None"
    """
    if pd.isna(text): 
        return "None"
    # Cleaning text data
    cleaned = str(text).replace("[", "").replace("]", "").replace("'", "")
    return cleaned.strip()

In [22]:
# --- HELPER: ClEANING STRING ---
def clean_string_data(text):
    """
    Sanitizes and formats raw text data from the dataset.
    
    This utility function is specifically designed to handle artifacts common in 
    CSV files, such as string representations of lists (e.g., "['Item']"), 
    and standardized missing value handling.

    ---------------------------------------------------------------------------
    Args:
        text (str, float, or None): 
            The raw input value to be cleaned. Can be a string, a list, or NaN.

    Returns:
        str: The cleaned, stripped string (e.g., "Dumbbell"). 
             Returns "None" if the input is null or NaN.
    
    ---------------------------------------------------------------------------
    Example:
        >>> clean_string_data("['Dumbbell']")
        'Dumbbell'
        >>> clean_string_data(np.nan)
        'None'
    """
    if pd.isna(text) or str(text).lower() == 'nan': return "None"
    return str(text).replace("[", "").replace("]", "").replace("'", "").strip()

# --- HELPER: CATEGORIZING TRAINING ---
def get_exercise_category(ex_name, muscle_group, env):
    """
    Classifies an exercise into a primary category (Sport, Cardio, or Strength) 
    based on environmental context and keywords.

    This function applies a priority-based heuristic to label the 'Workout_Type'. 
    It checks conditions in a specific order to ensure accurate categorization.

    ---------------------------------------------------------------------------
    Args:
        ex_name (str): 
            The name of the exercise (e.g., "Bench Press", "Jogging").
        muscle_group (str): 
            The target muscle group (e.g., "Chest", "Cardio").
        env (str): 
            The environment where the exercise is performed (e.g., "Home", "Gym", "Other").

    Returns:
        str: One of the following categories:
            - 'Sport'   : If the environment implies outdoor/sport activities (e.g., 'Other').
            - 'Cardio'  : If the exercise or muscle group explicitly mentions 'cardio'.
            - 'Strength': Default category for all other resistance or bodyweight exercises.
    ---------------------------------------------------------------------------
    Logic Priority:
    1. Environment Check (Sport) -> 2. Keyword Check (Cardio) -> 3. Default (Strength)
    """
    ex_name = str(ex_name).lower()
    muscle_group = str(muscle_group).lower()
    env = str(env)
    
    # 1. Sport
    if env == 'Other' or 'sport' in env.lower():
        return 'Sport'
    
    # 2. Cardio
    if 'cardio' in muscle_group or 'cardio' in ex_name:
        return 'Cardio'
    
    # 3. Strength / Weightlifting
    # Default for weightlifting
    return 'Strength'

# --- HELPER: SMART SEARCH WITH EXCLUSION ---
def get_exercises_smart(df_exercises, target_group, target_env, exclude_names=[], limit=1):
    """
    Intelligently retrieves exercise recommendations using a 3-Tier Hierarchical Search.

    This function attempts to find the best possible match for a target muscle group 
    and environment. If an exact match isn't available (e.g., due to equipment limits 
    at Home), it gracefully degrades to broader categories to ensure the user always 
    gets a valid workout.

    ---------------------------------------------------------------------------
    Args:
        df_exercises (pd.DataFrame): 
            The master exercise database containing columns ['targetMuscles', 'Environment', 'name'].
        target_group (str): 
            The specific muscle group to target (e.g., 'Quads', 'Chest').
        target_env (str): 
            The user's environment ('Home' or 'Gym').
        exclude_names (list, optional): 
            A list of exercise names already selected for this session (to prevent duplicates).
            Defaults to empty list [].
        limit (int, optional): 
            The maximum number of exercises to return. Defaults to 1.

    Returns:
        pd.DataFrame: A subset of the original dataframe containing the selected exercise(s).
                      Returns an empty DataFrame if absolutely no match is found (rare).

    ---------------------------------------------------------------------------
    Search Logic (Priority Order):

    1. Tier 1: Specific & Fresh (Best Case)
       Searches for exercises that strictly match the `target_group` AND have not 
       been used yet (`exclude_names`).
       Example: User wants 'Quads' -> Returns 'Goblet Squat'.

    2. Tier 2: General Fallback (Broad Match)
       If Tier 1 is empty (e.g., no isolated 'Calves' exercise available at 'Home'),
       it maps the specific group to a broader category.
       Example: User wants 'Calves' -> Database empty -> Returns a general 'Legs' exercise.

    3. Tier 3: Panic Mode (Duplicate Allowance)
       If both Tier 1 and Tier 2 fail, the function ignores the `exclude_names` list 
       and re-uses a valid exercise.
       Reasoning: It is better to repeat a 'Push Up' than to give the user 
       an empty workout slot.
    """

    # Filter Environment
    df_env = df_exercises[df_exercises['Environment'] == target_env]
    if df_env.empty: return pd.DataFrame()

    keywords = MUSCLE_GROUPS.get(target_group, [])
    
    # Filter Keyword
    mask_specific = df_env['targetMuscles'].astype(str).apply(
        lambda x: any(k.lower() in x.lower() for k in keywords)
    )
    
    # Exclude used
    mask_unused = ~df_env['name'].isin(exclude_names)
    
    # Trying to find unused and specific
    result = df_env[mask_specific & mask_unused]
    
    # FALLBACK LOGIC
    if result.empty:
        # Mapping Fallback
        fallback_map = {
            'Quads': 'Legs_General', 'Hamstrings': 'Legs_General', 'Glutes': 'Legs_General', 'Calves': 'Legs_General',
            'Chest': 'Push_General', 'Shoulders': 'Push_General', 'Triceps': 'Push_General',
            'Back': 'Pull_General', 'Biceps': 'Pull_General'
        }
        if target_group in fallback_map:
            gen_group = fallback_map[target_group]
            gen_keywords = MUSCLE_GROUPS.get(gen_group, [])
            mask_general = df_env['targetMuscles'].astype(str).apply(
                lambda x: any(k.lower() in x.lower() for k in gen_keywords)
            )
            result = df_env[mask_general & mask_unused]
    
    # PANIC MODE (Ambil Duplikat jika terpaksa)
    if result.empty:
        result = df_env[mask_specific]
        
    if len(result) > 0:
        return result.sample(n=min(limit, len(result)), replace=False)
    
    return pd.DataFrame()

## Generating Weekly Plan Function

In [23]:
def generate_weekly_plan(user_row, df_exercises):
    """
    Generates a precision-timed weekly workout schedule based on user constraints.

    This function is the core engine of the recommendation system. It constructs a 
    day-by-day itinerary that strictly respects the user's available time (Duration) 
    and workout frequency.

    ---------------------------------------------------------------------------
    Args:
        user_row (pd.Series): 
            A single row from the user profile dataframe containing:
            - 'Goal_x', 'Workout_Frequency_x', 'Average_Duration_Minutes_x'
            - 'level_x'
            - Sports flags (e.g., 'Badminton', 'Swim')
        df_exercises (pd.DataFrame): 
            The master database of exercises.

    ---------------------------------------------------------------------------
    Algorithmic Logic:

    1. PRECISION VOLUME CALCULATION (The "Duration" Logic):
       - Instead of guessing, we calculate the exact number of exercises a user 
         can fit into their session.
       - Formula: Total Time / (Sets * 3 mins per set).
       - Example: 60 mins / (4 sets * 3 mins) = 5 Exercises.
       - This ensures the generated plan is realistic and feasible.

    2. ADAPTIVE SPLIT MAPPING:
       - 1-2 Days: Full Body splits (Efficiency focus).
       - 3 Days  : Push / Pull / Legs (Classic split).
       - 4 Days  : Upper / Lower split (Strength focus).
       - 5-6 Days: Body Part Split (Hypertrophy focus).

    3. DYNAMIC WORKOUT STYLING:
       - Randomly assigns a "Style" (e.g., Cardio Focused, Strength Focused, Hybrid) 
         based on the user's Goal to provide variety.
       - Example: 'Weight Loss' users have a higher chance of getting 'Cardio Focused' days.

    4. SMART CARDIO & SPORT INJECTION:
       - If the user plays sports (e.g., Badminton), the algorithm occasionally 
         swaps a generic Cardio slot with their specific sport.
       - Adjusts strength slots dynamically to make room for cardio without 
         exceeding the total duration.

    ---------------------------------------------------------------------------
    Returns:
        pd.DataFrame: A detailed weekly schedule containing:
            ['User_ID', 'Day', 'Muscle Group', 'Exercise Name', 'Sets', 'Reps', 
             'Equipment', 'Instructions', 'Environment', 'Workout_Type']
    """
    
    user_id = user_row['User_ID']
    freq = user_row['Workout_Frequency_x'] 
    duration = user_row['Average_Duration_Minutes_x'] # e.g., 30, 45, 60, 90
    goal = user_row['Goal_x']
    level = user_row.get('level_x', 'Beginner')
    
    # --- A. DETERMINE SETS & REPS STRATEGY ---
    # Establish the base volume pattern to calculate time slots accurately.
    
    estimated_sets = 3 # Default Baseline
    
    # Adjust Sets based on Goal & Duration capability
    if goal == 'Muscle Gain': 
        # Hypertrophy benefits from volume, but only if time permits
        if duration >= 60: estimated_sets = 4
        else: estimated_sets = 3
    elif goal == 'Weight Loss': 
        # Weight loss often uses higher volume/reps to burn calories
        if duration >= 60: estimated_sets = 4
        else: estimated_sets = 3
    else: 
        estimated_sets = 3
        
    # Safety Constraint: Short duration sessions cannot support high volume
    if duration <= 30: estimated_sets = 3
    
    # --- B. ESTIMATE TIME PER EXERCISE ---
    # Standard assumption: 1 Set takes ~3 minutes (Performance + Rest)
    time_per_exercise = estimated_sets * 3 
    
    # --- C. CALCULATE TARGET EXERCISE COUNT (VOLUME) ---
    # This is the dynamic divider.
    # Example: 60 mins available / 12 mins per exercise = 5 Exercises total.
    target_exercises = int(duration / time_per_exercise)
    
    # Apply Safety Limits (Min 3 exercises, Max 12 exercises)
    target_exercises = max(3, target_exercises) 
    target_exercises = min(12, target_exercises)

    # --- D. SETUP AUXILIARY VARIABLES ---
    # Randomly assign a primary environment context
    main_env = random.choice(['Gym', 'Home'])
    
    # Extract user's sports capabilities
    user_sports = []
    # Assuming SPORTS_COLUMNS is defined globally or imported
    for col in SPORTS_COLUMNS:
        if user_row.get(col, 0) == 1:
            # Clean string: 'Badminton_x' -> 'Badminton'
            user_sports.append(col.replace('_x', '').replace('_', ' '))

    weekly_schedule = []
    
    # --- E. DEFINE WORKOUT STYLE (VARIETY) ---
    # Probabilistic assignment of workout focus based on Goal
    workout_style = 'Balanced' 
    if goal == 'Weight Loss':
        rand = random.random()
        workout_style = 'Cardio Focused' if rand < 0.4 else 'Strength Focused' if rand < 0.7 else 'Balanced'
    elif goal == 'Muscle Gain':
        workout_style = 'Pure Strength' if random.random() < 0.6 else 'Hybrid Athlete'
    
    # --- F. DEFINE SPLIT MAPPING (ROUTINE STRUCTURE) ---
    schedule_map = {}
    if freq <= 2:
        schedule_map = {
            1: {'Theme': 'Full Body Push', 'Focus': ['Quads', 'Chest', 'Shoulders', 'Triceps', 'Cardio']}, 
            2: {'Theme': 'Full Body Pull', 'Focus': ['Hamstrings', 'Back', 'Biceps', 'Glutes', 'Abs']}
        }
    elif freq == 3:
        schedule_map = {
            1: {'Theme': 'Push & Burn', 'Focus': ['Chest', 'Shoulders', 'Triceps', 'Cardio']}, 
            2: {'Theme': 'Pull & Core', 'Focus': ['Back', 'Biceps', 'Traps', 'Abs']}, 
            3: {'Theme': 'Leg Power', 'Focus': ['Quads', 'Hamstrings', 'Glutes', 'Calves']}
        }
    elif freq == 4:
        schedule_map = {
            1: {'Theme': 'Upper Strength', 'Focus': ['Chest', 'Back', 'Shoulders', 'Abs']}, 
            2: {'Theme': 'Lower Quads', 'Focus': ['Quads', 'Calves', 'Cardio', 'Abs']}, 
            3: {'Theme': 'Upper Pump', 'Focus': ['Biceps', 'Triceps', 'Chest', 'Back']}, 
            4: {'Theme': 'Lower Hams', 'Focus': ['Hamstrings', 'Glutes', 'Calves']}
        }
    else: 
        schedule_map = {
            1: {'Theme': 'Chest & Back', 'Focus': ['Chest', 'Back', 'Abs']}, 
            2: {'Theme': 'Legs & Cardio', 'Focus': ['Quads', 'Hamstrings', 'Cardio']}, 
            3: {'Theme': 'Shoulders & Arms', 'Focus': ['Shoulders', 'Biceps', 'Triceps']}, 
            4: {'Theme': 'Lower Glutes', 'Focus': ['Glutes', 'Calves', 'Abs']}, 
            5: {'Theme': 'Upper Mix', 'Focus': ['Chest', 'Shoulders', 'Back']}, 
            6: {'Theme': 'Cardio & Core', 'Focus': ['Abs', 'Cardio', 'Obliques']}
        }

    # --- G. GENERATION LOOP (DAY BY DAY) ---
    for day_num, config in schedule_map.items():
        if day_num > freq: break
        
        theme = config['Theme']
        day_muscles = list(config['Focus'])
        used_exercises_today = [] 

        # --- G.1. CARDIO INJECTION LOGIC ---
        # Modify the muscle list to include Cardio based on probabilities
        has_cardio = False
        
        if workout_style == 'Cardio Focused':
            day_muscles = ['Cardio', 'Cardio', 'Abs', 'Cardio', 'Cardio', 'Abs']
            has_cardio = True
        else:
            # Check probability for bonus cardio
            add_bonus = False
            if workout_style == 'Pure Strength' and random.random() < 0.2: add_bonus = True
            elif workout_style == 'Strength Focused' and random.random() < 0.5: add_bonus = True
            elif (workout_style == 'Hybrid Athlete' or workout_style == 'Balanced') and random.random() < 0.7: add_bonus = True
            
            if add_bonus and 'Cardio' not in day_muscles:
                # Append cardio to the end (accepting slight duration increase)
                day_muscles.append('Cardio') 
                has_cardio = True

        # --- G.2. SLOT DISTRIBUTION ---
        # Distribute the total available exercise slots among the target muscle groups
        num_groups = len(day_muscles)
        base_slot = target_exercises // num_groups
        remainder = target_exercises % num_groups
        
        for i, muscle in enumerate(day_muscles):
            my_limit = base_slot
            # Distribute remainder slots to the first few muscle groups
            if i < remainder: my_limit += 1
            my_limit = max(1, my_limit)
            
            if muscle == 'Cardio':
                # --- G.3. CARDIO SELECTION ---
                # Calculate cardio duration to match the time slot of a strength exercise
                # e.g., if 1 exercise takes 12 mins, cardio should be ~12-15 mins
                cardio_duration = time_per_exercise + random.randint(0, 5) 
                
                # Check if we should assign a specific Sport (Badminton, Swim, etc.)
                use_sport = False
                if len(user_sports) > 0 and random.random() < 0.7: use_sport = True
                
                if use_sport:
                    sport = random.choice(user_sports)
                    weekly_schedule.append({
                        'User_ID': user_id, 'Day': f"Day {day_num} - {theme}",
                        'Muscle Group': 'Cardio', 'Exercise Name': sport,
                        'Equipment': "['Sport Equipment']", 'SecondaryMuscles': "['None']",
                        'Sets': 1, 'Reps': f"{cardio_duration} Mins",
                        'Instructions': f"Play {sport}.", 'Environment': 'Other', 'Workout_Type': 'Sport'
                    })
                else:
                    # Get generic cardio
                    exercises = get_exercises_smart(df_exercises, 'Cardio', main_env, used_exercises_today, limit=1)
                    for _, ex in exercises.iterrows():
                        weekly_schedule.append({
                            'User_ID': user_id, 'Day': f"Day {day_num} - {theme}",
                            'Muscle Group': 'Cardio', 'Exercise Name': ex['name'],
                            'Equipment': clean_string_data(ex['equipments']),
                            'SecondaryMuscles': clean_string_data(ex.get('secondaryMuscles', "['None']")),
                            'Sets': 1, 'Reps': f"{cardio_duration} Mins",
                            'Instructions': clean_string_data(ex['instructions']), 
                            'Environment': ex['Environment'], 'Workout_Type': 'Cardio'
                        })
                        used_exercises_today.append(ex['name'])

            else:
                # --- G.4. STRENGTH SELECTION ---
                exercises = get_exercises_smart(df_exercises, muscle, main_env, used_exercises_today, limit=my_limit)
                for _, ex in exercises.iterrows():
                    sets = estimated_sets
                    
                    # Determine Rep Range based on Goal
                    reps = "8-12"
                    if goal == 'Weight Loss': reps = "12-15"
                    
                    category = get_exercise_category(ex['name'], muscle, ex['Environment'])
                    weekly_schedule.append({
                        'User_ID': user_id, 'Day': f"Day {day_num} - {theme}",
                        'Muscle Group': muscle, 'Exercise Name': ex['name'],
                        'Equipment': clean_string_data(ex['equipments']),
                        'SecondaryMuscles': clean_string_data(ex.get('secondaryMuscles', "['None']")),
                        'Sets': sets, 'Reps': reps,
                        'Instructions': clean_string_data(ex['instructions']), 
                        'Environment': ex['Environment'], 'Workout_Type': category 
                    })
                    used_exercises_today.append(ex['name'])

    return pd.DataFrame(weekly_schedule)

In [24]:
# --- EXECUTE ---
all_user_plans = []
unique_users_df = df_merged.drop_duplicates(subset=['User_ID'])

print("Loading Variation (Min 5 Moves, Random Env)...")

for index, user_row in unique_users_df.iterrows():
    # Call Fucntion
    my_plan = generate_weekly_plan(user_row, dataset_workout) 
    
    if not my_plan.empty:
        all_user_plans.append(my_plan)

if len(all_user_plans) > 0:
    df_final = pd.concat(all_user_plans, ignore_index=True)
    print(f"Success! Total schedule row: {len(df_final)}")
    
    # Preview for checking
    cols = ['User_ID', 'Day', 'Exercise Name', 'Environment', 'Workout_Type']
    print("\nPreview Data:")
    display(df_final[cols].head(15))
else:
    print("Warning: No Schedule!")

Loading Variation (Min 5 Moves, Random Env)...
Success! Total schedule row: 12451

Preview Data:


,User_ID,Day,Exercise Name,Environment,Workout_Type
0,1,Day 1 - Upper Strength,diamond push up,Home,Strength
1,1,Day 1 - Upper Strength,decline push up,Home,Strength
2,1,Day 1 - Upper Strength,superman,Home,Strength
3,1,Day 1 - Upper Strength,commando plank,Home,Strength
4,1,Day 1 - Upper Strength,flutter kicks,Home,Strength
5,1,Day 2 - Lower Quads,bodyweight squat,Home,Strength
6,1,Day 2 - Lower Quads,bulgarian split squat,Home,Strength
7,1,Day 2 - Lower Quads,calf jump,Home,Strength
8,1,Day 2 - Lower Quads,Burpees,Home,Cardio
9,1,Day 2 - Lower Quads,dead bug,Home,Strength


In [25]:
df_final_workout = pd.concat(all_user_plans, ignore_index=True)
df_final = pd.merge(
    df_merged,      # Weekly Data (Week 0-12)
    df_final_workout,   # Wokrkout Data (Day 1-X)
    on='User_ID',   # Connecting Key
    how='left'      # Left Join
)

print(f"Success! df_final created with size: {df_final.shape}")

Success! df_final created with size: (12451, 45)


In [26]:
# Column List to be cleaned (duplicated)
cols_to_clean = [
    'Badminton', 'Football', 'Basketball', 'Volleyball', 'Swim',
]

for col in cols_to_clean:
    col_x = f"{col}_x"
    col_y = f"{col}_y"
    
    # checking if the data in our dataframe
    if col_x in df_final.columns and col_y in df_final.columns:
        # we keep _x for final data
        df_final.rename(columns={col_x: col}, inplace=True)
        # delete _y
        df_final.drop(columns=[col_y], inplace=True)
        
    # Fallback safety if there is only _x
    elif col_x in df_final.columns:
        df_final.rename(columns={col_x: col}, inplace=True)

## Calculate Workout Details Function

In [27]:
def calculate_workout_details(row):
    """
    Computes precise workout metrics by splitting the session into 'Active' and 'Rest' phases.

    This function calculates three key performance indicators (KPIs) for each exercise entry:
    1. Caloric Burn: Based on the Total Session Time (Active + Rest) using standard MET values.
    2. Active Duration: The actual time spent performing the movement (Time Under Tension).
    3. Rest Duration: The time spent recovering between sets.

    ---------------------------------------------------------------------------
    Args:
        row (pd.Series): A single row from the schedule dataframe containing:
            - 'Weight_kg', 'Workout_Type', 'Exercise Name'
            - 'Sets', 'Reps'

    ---------------------------------------------------------------------------
    Logic & Formulas:

    A. STRENGTH TRAINING (Weightlifting):
       - Assumption: 1 Set takes approximately 3 minutes total.
       - Split: 
         • 1 Minute Active (Lifting/Lowering).
         • 2 Minutes Rest (Recovery).
       - Total Time = Sets * 3 minutes.

    B. CARDIO & SPORTS:
       - Continuous Cardio (e.g., Running, Swimming): 
         • Active Time = 100% of duration.
         • Rest Time = 0.
       - Interval Training:
         • Active Time = 50% of duration.
         • Rest Time = 50% of duration.

    C. CALORIE CALCULATION (METs):
       - Uses Metabolic Equivalent of Task (MET) standardized values.
       - Formula: Calories = MET * Weight(kg) * Duration(hours).
       - Note: Duration used here is the TOTAL session time to reflect real-world energy expenditure.

    ---------------------------------------------------------------------------
    Returns:
        pd.Series: [Calories_Burned (int), Active_Minutes (int), Rest_Minutes (int)]
    """
    
    # --- 1. DATA EXTRACTION ---
    # Safe retrieval with fallbacks for missing data
    weight = row.get('Weight_kg')
    if pd.isna(weight): weight = row.get('Initial_Weight_kg_x', 60)

    workout_type = str(row.get('Workout_Type', 'Strength'))
    level = str(row.get('level_x', 'Beginner'))
    exercise_name = str(row.get('Exercise Name', '')).lower()
    
    # Initialize Accumulators
    active_minutes = 0       # Pure Movement Time
    rest_minutes = 0         # Recovery Time
    total_session_time = 0   # Total Duration (Active + Rest) -> Used for Calorie Math
    
    # --- 2. DURATION CALCULATION (ACTIVE vs REST) ---
    
    # A. STRENGTH TRAINING LOGIC
    if workout_type == 'Strength':
        try:
            sets = int(row.get('Sets', 3))
            
            # SPLIT LOGIC:
            # We estimate 1 Set = 3 Minutes Total
            # - 1 Minute: Time Under Tension (Active)
            # - 2 Minutes: Rest Period
            
            active_minutes = sets * 1  
            rest_minutes = sets * 2    
            
            total_session_time = active_minutes + rest_minutes 
            
        except:
            # Fallback values if data is corrupted
            active_minutes = 4
            rest_minutes = 8
            total_session_time = 12
            
    # B. CARDIO / SPORT LOGIC
    else:
        try:
            # Parse duration from Reps string (e.g., "30 Mins")
            reps_str = str(row.get('Reps', '30 Mins'))
            total_cardio_time = 30 # Default
            
            if 'Min' in reps_str:
                # Extract number from string
                total_cardio_time = int(reps_str.replace('Mins', '').replace('Min', '').strip())
            
            # Handle Interval Training vs Continuous
            if 'interval' in exercise_name:
                # Assume 1:1 Work/Rest Ratio
                active_minutes = int(total_cardio_time * 0.5)
                rest_minutes = total_cardio_time - active_minutes
            else:
                # Continuous Effort (Running, Swimming, Sports)
                # Entire duration is considered Active
                active_minutes = total_cardio_time
                rest_minutes = 0
            
            total_session_time = total_cardio_time
                
        except:
            # Fallback
            active_minutes = 30
            rest_minutes = 0
            total_session_time = 30

    # --- 3. CALORIE CALCULATION (METs METHOD) ---
    # We use Total Time because METs usually account for the pacing of a full session
    duration_hour = total_session_time / 60
    met = 3.5 # Default MET (Light activity)
    
    # Assign MET based on Type and Intensity
    if workout_type == 'Strength':
        if level == 'Beginner': met = 3.5      # Light weightlifting
        elif level == 'Intermediate': met = 5.0 # Moderate effort
        else: met = 6.0                        # Vigorous lifting
        
    elif workout_type == 'Cardio' or workout_type == 'Sport':
        # Specific METs for common activities
        if 'run' in exercise_name: 
            met = 9.8 if 'sprint' in exercise_name else 9.0
        elif 'walk' in exercise_name: met = 4.3
        elif 'badminton' in exercise_name: met = 5.5
        elif 'football' in exercise_name: met = 8.0
        elif 'swim' in exercise_name: met = 8.0
        elif 'cycle' in exercise_name: met = 7.5
        else: met = 6.0 

    # Calculate Total Burn
    calories = int(met * weight * duration_hour)
    
    # Return strict format for Pandas apply()
    return pd.Series([calories, active_minutes, rest_minutes])

# --- Execution ---
print("Recalculating...")

cols = ['Calories_Burned', 'Duration_Minutes', 'Rest_Minutes']
df_final[cols] = df_final.apply(calculate_workout_details, axis=1)
print("Success!")

Recalculating...
Success!


## Saving Dataset

In [28]:
output_file = '../../data/dataset_workout_final.csv'

# Saving
df_final.to_csv(output_file, index=False)

print(f"Data Saved to: {output_file}")
print("Total Row:", len(df_final))

Data Saved to: ../../data/dataset_workout_final.csv
Total Row: 12451
